# 环节 04 · Attention（配套 Notebook）

> 配套长文：[环节04-Attention注意力详解.md](./环节04-Attention注意力详解.md)
> 定位：把长文 §5 的**全数值手算走查**一行一行跑出来（纯 Python，不用 numpy），再补上缩放、Mask、多头、KV Cache 与 FlashAttention 的可运行验证。

**怎么跑**：逐格 `Shift+Enter`；后面的格子依赖前面已执行的变量。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 单头流程 | §2 | Q/K/V → 打分 → Mask → softmax → 加权求和 |
| §2 数值走查 | §5 | 复现手算的 P / Z（含 loss，衔接环节08） |
| §3 为什么要除 √d_k | §2.2 | 不缩放会 softmax 饱和 |
| §4 因果 Mask | §2.3 | 「并行算全序列」≡「逐词单算」 |
| §5 多头 | §2.4 | 切维、各头注意力图不同、concat 回全维 |
| §6 KV Cache 与家族 | §3 / §4 | MHA / GQA / MQA 的显存账 |
| §7 FlashAttention | §3 | 分块 + 在线 softmax 与朴素版结果一致 |
| §8 O(n²) 的规模 | §4 | 长上下文的"税"到底多大 |


## 1. 单头流程（长文 §2）

```
Q = X·W_Q,   K = X·W_K,   V = X·W_V          # 三个投影
S = Q·Kᵀ / √d_k                              # 打分：(n, n) ← O(n²) 的来源
M[i][j] = -∞ if j > i else 0                 # 因果 Mask（上三角）
P = softmax(S + M)（按行）                    # 注意力权重，每行和为 1
O = P·V                                      # 按相关度加权的内容求和
```

先用纯 Python 造好最小工具（矩阵乘、转置、softmax）——不用 numpy，每一步都能看清楚：


In [ ]:
import math


def matmul(A, B):
    """矩阵乘 A·B（B 转成列再点积）。"""
    cols = list(zip(*B))
    return [[sum(a * b for a, b in zip(row, col)) for col in cols] for row in A]


def transpose(A):
    return [list(col) for col in zip(*A)]


def softmax(row):
    """按行 softmax；-inf 位置的权重会被挤成 0（这正是 Mask 生效的方式）。"""
    m = max(x for x in row if x != float("-inf"))
    exps = [0.0 if x == float("-inf") else math.exp(x - m) for x in row]
    s = sum(exps)
    return [e / s for e in exps]


def show(title, M, row_names=None, digits=5):
    print(f"\n{title}")
    for i, row in enumerate(M):
        name = f"{row_names[i]:>6}  " if row_names else "   "
        print(name + " ".join(f"{v:>{digits + 4}.{digits}f}" for v in row))


print("工具就绪：matmul / transpose / softmax")


## 2. 数值走查：复现长文 §5 的全部数字

玩具设定（与长文完全一致）：单层、单头、`d = 4`、词表 `{a,b,c,d}`：

```
输入 X = [a, b, c]        标签 Y = [b, c, d]
Embedding：a=[1,0,0,0]  b=[0,1,0,0]  c=[0,0,1,0]
W_Q = W_K = W_V = I        缩放 √d_k = 2
```


In [ ]:
TOKENS = ["a", "b", "c"]
X = [
    [1.0, 0.0, 0.0, 0.0],      # a
    [0.0, 1.0, 0.0, 0.0],      # b
    [0.0, 0.0, 1.0, 0.0],      # c
]
Y = [1, 2, 3]                  # 位置0→b、位置1→c、位置2→d
n, dk = len(X), len(X[0])

# Step 2：本例 W_Q = W_K = W_V = I，所以 Q = K = V = X
Q = K = V_ = [row[:] for row in X]

S_raw = [[sum(Q[i][k] * K[j][k] for k in range(dk)) / math.sqrt(dk)
          for j in range(n)] for i in range(n)]
show("Step 2 · 原始注意力分数 S_raw = Q·Kᵀ/√d_k", S_raw, TOKENS)
print("（各词嵌入两两正交、自相似为 1 → 对角线 1/2 = 0.5，跨词全 0）")

# Step 3：因果 Mask，把上三角（j > i）置 -inf
S = [[float("-inf") if j > i else S_raw[i][j] for j in range(n)] for i in range(n)]
show("Step 3 · 因果 Mask 后的 S（-inf 即“看不见”）", S, TOKENS)
print("注意 -inf 只是个记号；下一步 softmax 会把它们压成 0。")


In [ ]:
# Step 4：softmax 按行归一 → 注意力权重 P
P = [softmax(row) for row in S]
show("Step 4 · 注意力权重 P（每行和 = 1）", P, TOKENS)
print(f"行和检查：{[round(sum(r), 6) for r in P]}")
print("\n对照长文 §5：行0 [1,0,0]；行1 [0.37754, 0.62246]；行2 [0.27407, 0.27407, 0.45186]")
print("（长文写 0.27408 / 0.45185，是四舍五入末位的 1e-5 差异，不是算法差异）")

# Step 5：加权求和 O = P·V
O = matmul(P, V_)
show("Step 5 · O = P·V（每个位置看得见的 token 的加权内容）", O, TOKENS, digits=5)

# Step 6：残差 Z = X + O
Z = [[X[i][k] + O[i][k] for k in range(dk)] for i in range(n)]
show("Step 6 · 残差 Z = X + O", Z, TOKENS, digits=5)
print("Z 同时带着“原始 token 信息 + 上下文加权信息”（残差详解见环节 06）。")


In [ ]:
# 走查的“下半场”：Z 过 LM Head 得到预测分布 → 交叉熵损失
# （长文说这一半在环节 08 §4，数字完全相同；这里先跑出来，方便对照）
logits = Z                                  # 本例 W_head = I
probs = [softmax(row) for row in logits]

print("预测分布（每行 = 该位置对下一个词的预测）：")
print("        " + "".join(f"{w:>9}" for w in ["a", "b", "c", "d"]))
for i, row in enumerate(probs):
    print(f"位置{i}  " + "".join(f"{p:>9.5f}" for p in row))

loss = sum(-math.log(probs[i][Y[i]]) for i in range(n)) / n
print(f"\n交叉熵 loss = {loss:.5f}")
for i in range(n):
    print(f"  位置{i}: 标签 = {['a','b','c','d'][Y[i]]}，预测概率 {probs[i][Y[i]]:.5f}"
          f" → -log = {-math.log(probs[i][Y[i]]):.4f}")
print("\n（长文注释写 ≈ 2.1838；差异来自它用四舍五入后的中间值代入，本格全程用精确值。）")
print("这就是“注意力输出 → 预测下一个词 → 计损失”的完整闭环，也是环节 09 训练管线的第一步。")


## 3. 为什么要除以 √d_k（长文 §2.2）

`d` 维随机向量的点积量级 ~ `√d`：不缩放的话，分数动辄 ±20，softmax 直接进入**饱和区（saturation）**——概率贴在 0/1 上，梯度接近消失。

用 `d = 64` 随机造 200 组「1 个 q 对 4 个 k」，比较两种情况：


In [ ]:
import random

random.seed(1)
TOPS, ENTS = {}, {}
for label, divide in [("不缩放", False), ("除以 √d_k", True)]:
    tops, ents = [], []
    for _ in range(200):
        q = [random.gauss(0, 1) for _ in range(64)]
        ks = [[random.gauss(0, 1) for _ in range(64)] for _ in range(4)]
        sc = [sum(a * b for a, b in zip(q, k)) for k in ks]
        if divide:
            sc = [s / math.sqrt(64) for s in sc]
        p = softmax(sc)
        tops.append(max(p))
        ents.append(-sum(x * math.log(x) for x in p if x))
    TOPS[label], ENTS[label] = sum(tops) / len(tops), sum(ents) / len(ents)

print(f"{'情况':<10} {'top1 平均概率':>14} {'平均熵':>10}   （均匀分布时熵 = {math.log(4):.4f}）")
for label in TOPS:
    print(f"{label:<10} {TOPS[label]:>14.4f} {ENTS[label]:>10.4f}")

print("\n→ 不缩放：概率几乎全压在第一名（0.91），熵只有 0.22 —— softmax 饱和，梯度被挤没；")
print("  除以 √d_k 后：分布回到正常量级（0.51 / 1.12），模型才有梯度可学。")


## 4. 因果 Mask 的真正作用（长文 §2.3）

训练时整条序列并行前向，却要求位置 i 只能看见 j ≤ i。做到这一点**不需要改前向逻辑**，只需把上三角置 −∞。

反过来说：**「并行算全序列」必须与「逐词单独算」结果完全一致**，否则训练和推理就对不上了。验证一下：


In [ ]:
print(f"{'位置':<6} {'并行整条序列':<34} {'只用前 i+1 个 token 单算':<34} 一致")
print("-" * 92)
for i in range(n):
    sub = X[:i + 1]                                    # 只取前 i+1 个 token（维度不变）
    Ss = [[sum(sub[a][k] * sub[b][k] for k in range(dk)) / math.sqrt(dk)
           for b in range(i + 1)] for a in range(i + 1)]
    Ps = [softmax(r) for r in Ss]
    Os = matmul(Ps, sub)
    same = all(abs(Os[i][k] - O[i][k]) < 1e-12 for k in range(dk))
    print(f"{i:<6} {str([round(v, 5) for v in O[i]]):<34} "
          f"{str([round(v, 5) for v in Os[i]]):<34} {same}")

print("\n→ 两者逐位相同。Mask 之外的那些 −∞ 概率为 0，既不贡献内容，也不会传梯度。")


## 5. 多头：切维、各看一科、最后汇总（长文 §2.4）

`d = 8`、`H = 2` → 每头 `d_head = 4`。每头只在**自己的窄子空间**里独立打分、独立加权，最后 `concat` 回全维再过 `W_O`。


In [ ]:
import random

random.seed(2)
n2, d2, H = 3, 8, 2
d_head = d2 // H
X2 = [[random.gauss(0, 1) for _ in range(d2)] for _ in range(n2)]

head_outs, head_attn = [], []
for h in range(H):
    Xh = [row[h * d_head:(h + 1) * d_head] for row in X2]        # 本头分到的切片
    Sh = [[sum(Xh[i][k] * Xh[j][k] for k in range(d_head)) / math.sqrt(d_head)
           for j in range(n2)] for i in range(n2)]
    Ph = [softmax(r) for r in Sh]
    head_attn.append(Ph)
    head_outs.append(matmul(Ph, Xh))
    show(f"头 {h + 1} 的注意力图（d_head={d_head} 子空间内打分）", Ph, digits=4)

O = [sum((head_outs[h][i] for h in range(H)), []) for i in range(n2)]     # concat
print(f"\n每头输出形状 {len(head_outs[0])}×{len(head_outs[0][0])}，"
      f"concat 后 {len(O)}×{len(O[0])} → 再过 W_O 融合回 (n, d)")
print(f"两个头的注意力图不同 = {head_attn[0] != head_attn[1]}")
print("\n算力账：两头各自 (n × d_head) 的子空间打分，总开销 ≈ 单头全维一次；")
print("“谁管语法、谁管指代”不是预设的 —— 是训练中反向传播自发分化出来的（长文 §2.4）。")


## 6. KV Cache 是怎么来的、家族怎么省（长文 §3 / §4）

自回归生成第 t 步要再看一遍前 t−1 个 token 的 K/V。**存下来不重算**就是 KV Cache：

```
KV Cache ≈ 2（K 和 V）× 层数 × KV 头数 × 头维度 × 总长度 × 字节数
```

GQA / MQA 砍的就是公式里 **KV 头数**这一项：


In [ ]:
LAYERS, D_HEAD, LEN, BYTES = 80, 128, 4096, 2      # 80 层、头维度 128、长度 4096、fp16

print(f"设定：{LAYERS} 层、d_head={D_HEAD}、序列长度 {LEN}、fp16")
print(f"{'方案':<24} {'KV 头数':>8} {'KV Cache':>12}   {'相对 MHA':>10}")
print("-" * 60)
base = None
for name, kv_heads in [("MHA（每头一套 K/V）", 64),
                       ("GQA（8 组共享，主流）", 8),
                       ("MQA（全部共享 1 套）", 1)]:
    gb = 2 * LAYERS * kv_heads * D_HEAD * LEN * BYTES / 1024 ** 3
    if base is None:
        base = gb
    print(f"{name:<24} {kv_heads:>8} {gb:>10.2f} GB {base / gb:>9.0f}x")

print("\n（MLA 是另一条路：先把 K/V 压进低维潜在空间再展开，显存再降一个量级 ——")
print("  结构更复杂，是 DeepSeek V2/V3 的路线。注意这一切都**没有改变每个头怎么算注意力**。）")
print("\n记忆钩子：H > G > 1。MHA → GQA → MQA → MLA，KV 越省、结构越复杂。")


## 7. FlashAttention 在干什么（长文 §3）

它不改结构，只改**怎么算**：不打那个巨大的 `(n, n)` 中间矩阵，而是**分块（blockwise）**算，配合**在线 softmax（online softmax）**——每处理完一块就用"当前最大值"和"当前累加和"修正前面已算的部分。

关键要求：结果必须和朴素 softmax **完全一致**。验证：


In [ ]:
def attention_naive(scores, V):
    """朴素做法：先算出完整 P，再乘 V。"""
    P = [softmax(r) for r in scores]
    return matmul(P, V)


def attention_flash(scores, V, block=2):
    """分块 + 在线 softmax：全程不需要物化完整的 (n, n) 概率矩阵。"""
    out = [[0.0] * len(V[0]) for _ in scores]
    for i, row in enumerate(scores):
        m_run, l_run = float("-inf"), 0.0      # running max / running sum
        acc = [0.0] * len(V[0])
        for start in range(0, len(row), block):
            blk = row[start:start + block]
            m_new = max(m_run, max(blk))
            corr = math.exp(m_run - m_new) if m_run != float("-inf") else 0.0
            acc = [a * corr for a in acc]       # 最大值变了，旧累加要按比例缩回来
            l_run *= corr
            for t, s in enumerate(blk):
                w = math.exp(s - m_new)
                l_run += w
                for k in range(len(V[0])):
                    acc[k] += w * V[start + t][k]
            m_run = m_new
        out[i] = [a / l_run for a in acc]
    return out


random.seed(3)
n3, d3 = 6, 2
scores = [[random.gauss(0, 1) for _ in range(n3)] for _ in range(n3)]
V3 = [[random.gauss(0, 1) for _ in range(d3)] for _ in range(n3)]

O_naive = attention_naive(scores, V3)
print(f"{'分块大小':>10} {'与朴素版最大差异':>20}   {'显存里存的中间量':>18}")
print("-" * 62)
for blk in (1, 2, 3, 6):
    O_flash = attention_flash(scores, V3, blk)
    diff = max(abs(a - b) for ra, rb in zip(O_naive, O_flash) for a, b in zip(ra, rb))
    print(f"{blk:>10} {diff:>20.2e}   {f'{n3}×{blk} 块':>18}")

print(f"\n→ 差异都在 {1e-16:.0e} 量级（浮点误差），结果一致。")
print("  朴素版要物化 n×n 的 P；分块版只用 n×block 的一小块 + 两个 running 标量。")
print("  这就是 FlashAttention “省显存又更快”的全部秘密（IO 感知）。")


## 8. O(n²) 的规模：长上下文的"税"（长文 §4）


In [ ]:
print(f"{'序列长度 n':>12} {'S/P 元素个数':>16} {'若物化成 fp16':>14} {'单层 32 头合计':>16}")
print("-" * 64)
for nn in (1024, 4096, 32768, 131072):
    elems = nn * nn
    gb = elems * 2 / 1024 ** 3
    print(f"{nn:>12,} {elems:>16,} {gb:>12.3f} GB {gb * 32:>14.1f} GB")

print("\n→ 长度翻倍，注意力矩阵大小翻 4 倍 —— 这就是 O(n²) 的来源。")
print("  注意后两列只是“如果真把矩阵存下来”的大小：现实中没人这么干，")
print("  正因为不可承受，才有 §7 的分块算法（FlashAttention）与各种稀疏/线性注意力。")
print("  另一笔账：KV Cache 随长度**线性**涨，长文本优化基本都在处理这两条曲线（环节 10/11）。")


## 9. 自测（长文 §6）

| 问题 | 本 Notebook 的现场证据 |
|---|---|
| Self-Attention 为什么能并行？ | §2：整条序列一次矩阵乘算完，各位置无串行依赖 |
| 为什么要除以 √d_k？ | §3：不缩放 top1 概率 0.91、熵 0.22（饱和） |
| 训练时凭什么不偷看未来？ | §4：上三角 −∞ → 概率 0；且并行结果 ≡ 逐词单算 |
| 多头凭什么更强还不加算力？ | §5：切维成 H 份子空间，总算力 ≈ 单头全维一次 |
| KV Cache 为什么是显存大头？ | §6：`2×层×KV头×d_head×长度×字节`，MHA 在 4096 长度下就 10 GB |
| GQA / MQA / MLA 在省什么？ | §6：砍 KV 头数；MLA 改成先压缩再展开 |
| FlashAttention 快在哪？ | §7：分块 + 在线 softmax，不物化 n×n，结果与朴素版一致（差 1e-16） |
| 长上下文为什么贵？ | §8：注意力 O(n²) + KV Cache 线性涨 |

**接续**：Z 之后怎么算损失、梯度怎么传，去 [环节 08 · 输出头与训练目标](./环节08-输出头与训练目标详解.md) §4（手算数字与本 Notebook §2 完全相同）。
